In [ ]:
%reload_ext autoreload

# Modell Tunining von Gradientboosting auf unseren Daten

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix
from scipy.stats import loguniform, randint
from utils import bewerten, trainieren_und_bewerten

---
## Daten laden

In [ ]:
from pathlib import Path

ordner = Path('../data/processed')

X_train = pd.read_parquet(ordner / 'X_train.parquet')
X_test = pd.read_parquet(ordner / 'X_test.parquet')
y_train = pd.read_parquet(ordner / 'y_train.parquet')['loan_status']
y_test = pd.read_parquet(ordner / 'y_test.parquet')['loan_status']


---
## Tuning beginnen

### Ungetuned

In [ ]:
model_gb = HistGradientBoostingClassifier(random_state=42)
model_gb.fit(X_train, y_train)
bewerten('Gradient Boosting (Standard)', y_test, model_gb.predict(X_test))

---
### Schritt 1

In [ ]:
param_dist = {
    'learning_rate': loguniform(0.01, 0.3),
    'max_leaf_nodes': randint(15, 64),
    'min_samples_leaf': randint(10, 100),
    'l2_regularization': loguniform(1e-3, 10),
}

suche = RandomizedSearchCV(
    HistGradientBoostingClassifier(max_iter=1000, early_stopping=True, random_state=42),
    param_distributions=param_dist,
    n_iter=40,
    cv=5,
    scoring='roc_auc',            # NEU: statt 'accuracy'
    return_train_score=True,      # NEU: für den Overfitting-Check
    n_jobs=-1,
    random_state=42,
    verbose=1,
)
suche.fit(X_train, y_train)

ergebnis = pd.DataFrame(suche.cv_results_).loc[suche.best_index_]
print('Beste Parameter:', suche.best_params_)
print(f"AUC Training: {ergebnis['mean_train_score']:.3f} | AUC Cross-Validation: {ergebnis['mean_test_score']:.3f}")

---
### Schritt 2


In [ ]:
model_gb_tuned = suche.best_estimator_
schwelle = 0.3   # ← HIER anpassen: kleiner = strenger

proba = model_gb_tuned.predict_proba(X_test)[:, 1] 
bewerten(f'Gradient Boosting (getunt, Schwelle {schwelle})', y_test, (proba >= schwelle).astype(int), proba)

zeilen = []
for s in [0.5, 0.4, 0.3, 0.2, 0.15, 0.1]:
    genehmigt = proba < s
    zeilen.append({
        'Schwelle': s,
        'Genehmigungsquote': genehmigt.mean(),
        'Ausfallquote genehmigt': y_test.values[genehmigt].mean(),
        'Gute Kunden abgelehnt': (~genehmigt & (y_test.values == 0)).sum() / (y_test.values == 0).sum(),
    })
display(pd.DataFrame(zeilen).style.format({c: '{:.1%}' for c in ['Genehmigungsquote', 'Ausfallquote genehmigt', 'Gute Kunden abgelehnt']}))

---
###